In [1]:
import os
import numpy as np
import cv2
from tqdm import tqdm

def load_org_images(folder_path):

    subfolders = sorted(os.listdir(folder_path))
    subfolders = [f for f in subfolders if os.path.isdir(os.path.join(folder_path, f))]

    first_img = cv2.imread(os.path.join(folder_path, subfolders[0], "0.jpg"), cv2.IMREAD_GRAYSCALE)
    H, W = first_img.shape

    data = np.zeros((len(subfolders), H, W), dtype=np.uint8)

    for i, folder in enumerate(tqdm(subfolders, desc="Loading org_images")):
        img_path = os.path.join(folder_path, folder, "0.jpg")
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        data[i] = img

    return data

In [2]:
def load_raw_images(folder_path):

    subfolders = sorted(os.listdir(folder_path))
    subfolders = [f for f in subfolders if os.path.isdir(os.path.join(folder_path, f))]

    first_img = cv2.imread(os.path.join(folder_path, subfolders[0], "0.png"), cv2.IMREAD_GRAYSCALE)
    H, W = first_img.shape

    data = np.zeros((len(subfolders), 49, H, W), dtype=np.uint8)

    for i, folder in enumerate(tqdm(subfolders, desc="Loading raw_images")):
        for j in range(49):
            img_path = os.path.join(folder_path, folder, f"{j}.png")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            data[i, j] = img

    return data

In [3]:
def load_depth_images(folder_path):

    subfolders = sorted(os.listdir(folder_path))
    subfolders = [f for f in subfolders if os.path.isdir(os.path.join(folder_path, f))]

    first_img = cv2.imread(os.path.join(folder_path, subfolders[0], "depth.png"), cv2.IMREAD_UNCHANGED)
    H, W = first_img.shape[:2]

    data = np.zeros((len(subfolders), H, W), dtype=np.uint8)

    for i, folder in enumerate(tqdm(subfolders, desc="Loading depth_images")):
        img_path = os.path.join(folder_path, folder, "depth.png")
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        data[i] = img.astype(np.uint8)

    return data

In [4]:
import OpenEXR
import Imath


def read_exr(path):
    exr_file = OpenEXR.InputFile(path)
    header = exr_file.header()
    dw = header['dataWindow']
    W = dw.max.x - dw.min.x + 1
    H = dw.max.y - dw.min.y + 1

    pt = Imath.PixelType(Imath.PixelType.FLOAT)
    channel = exr_file.channel('R', pt)

    img = np.frombuffer(channel, dtype=np.float32)
    img = img.reshape(H, W)

    img = np.clip(img, 0.0, 1.0)

    return img


def load_conf_images(folder_path):

    subfolders = sorted(os.listdir(folder_path))
    subfolders = [f for f in subfolders if os.path.isdir(os.path.join(folder_path, f))]

    first_img = read_exr(os.path.join(folder_path, subfolders[0], "conf.exr"))
    H, W = first_img.shape

    data = np.zeros((len(subfolders), H, W), dtype=np.float32)

    for i, folder in enumerate(tqdm(subfolders, desc="Loading conf_images")):
        img_path = os.path.join(folder_path, folder, "conf.exr")
        data[i] = read_exr(img_path)

    return data

In [5]:
import os
import json

def write_to_memmaps(
    org_folder,
    raw_folder,
    depth_folder,
    conf_folder,
    org_path,
    raw_path,
    depth_path,
    conf_path,
    meta_path   
):

    metadata = {}

    # ---------- ORG ----------
    org_data = load_org_images(org_folder)
    org_mm = np.memmap(org_path, dtype=np.uint8, mode='w+', shape=org_data.shape)
    org_mm[:] = org_data
    org_mm.flush()

    metadata["org"] = {
        "shape": org_data.shape,
        "dtype": "uint8"
    }

    del org_data

    # ---------- RAW ----------
    raw_data = load_raw_images(raw_folder)
    raw_mm = np.memmap(raw_path, dtype=np.uint8, mode='w+', shape=raw_data.shape)
    raw_mm[:] = raw_data
    raw_mm.flush()

    metadata["raw"] = {
        "shape": raw_data.shape,
        "dtype": "uint8"
    }

    del raw_data

    # ---------- DEPTH ----------
    depth_data = load_depth_images(depth_folder)
    depth_mm = np.memmap(depth_path, dtype=np.uint8, mode='w+', shape=depth_data.shape)
    depth_mm[:] = depth_data
    depth_mm.flush()

    metadata["depth"] = {
        "shape": depth_data.shape,
        "dtype": "uint8"
    }

    del depth_data

    # ---------- CONF ----------
    conf_data = load_conf_images(conf_folder)
    conf_mm = np.memmap(conf_path, dtype=np.float32, mode='w+', shape=conf_data.shape)
    conf_mm[:] = conf_data
    conf_mm.flush()

    metadata["conf"] = {
        "shape": conf_data.shape,
        "dtype": "float32"
    }

    del conf_data

    # ---------- Save metadata ----------
    with open(meta_path, "w") as f:
        json.dump(metadata, f)

In [6]:
dataset_type = "Train"

data_path = "/mnt/Velocity_Vault/Datasets/Autofocus/"+dataset_type
memmap_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Memory/"+dataset_type


org_folder = data_path+"/scaled_images"
raw_folder = data_path+"/raw_up_pd"
depth_folder = data_path+"/merged_depth"
conf_folder = data_path+"/merged_conf"

org_path = memmap_path +"/org_images.mm"
raw_path = memmap_path +"/raw_images.mm"
depth_path = memmap_path +"/depth_images.mm"
conf_path = memmap_path +"/conf_images.mm"

meta_path = memmap_path +"/meta_data.txt"


write_to_memmaps(
    org_folder,
    raw_folder,
    depth_folder,
    conf_folder,
    org_path,
    raw_path,
    depth_path,
    conf_path,
    meta_path
)

Loading conf_images: 100%|██████████| 354/354 [00:01<00:00, 297.67it/s]
